# STEP 05: Model Evaluation & Serialization

This notebook evaluates trained models on test set predictions, prints the final evaluation results table, selects the top model, and serializes the pipeline to `../ml-service/model/best_model.pkl`.

In [1]:
# Import evaluation dependencies and load dataset
import pandas as pd
import numpy as np
import joblib
import os
import warnings
warnings.filterwarnings('ignore')
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, TargetEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, VotingRegressor

df = pd.read_csv("../dataset/processed/engineered_house_rent_dataset.csv")
X = df.drop(columns=["Rent"])
y = df["Rent"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

In [2]:
# Evaluation helper function to calculate MAE, RMSE, and R2 metrics
def evaluate_model(name, y_true, predictions):
    mae = mean_absolute_error(y_true, predictions)
    rmse = np.sqrt(mean_squared_error(y_true, predictions))
    r2 = r2_score(y_true, predictions)
    return {"Model": name, "MAE": mae, "RMSE": rmse, "R2": r2}

In [3]:
# Preprocessor definition matching ml-service pipeline
numeric_features = [
    "BHK", "Size", "Bathroom", "Current_Floor", "Total_Floors",
    "Floor_Ratio", "Bathroom_BHK_Ratio", "Size_Per_BHK", "Size_Per_Bathroom",
    "Is_Top_Floor", "Is_Ground_Floor", "Posted_Year", "Posted_Month", "Posted_DayOfWeek", "Log_Size"
]
categorical_features = [
    "Area Type", "Area Locality", "City", "Furnishing Status", "Tenant Preferred", "Size_Category",
    "City_Locality", "City_BHK", "City_Furnishing"
]

preprocessor = ColumnTransformer(transformers=[
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), numeric_features),
    ("cat", TargetEncoder(smooth="auto", cv=5, random_state=42), categorical_features)
])

rf = RandomForestRegressor(n_estimators=800, max_depth=18, min_samples_split=2, min_samples_leaf=1, max_features=0.7, random_state=42, n_jobs=-1)
xgb_m = xgb.XGBRegressor(
    n_estimators=1500, learning_rate=0.012, max_depth=6, subsample=0.75, colsample_bytree=0.7,
    min_child_weight=3, gamma=0.05, reg_alpha=0.1, reg_lambda=1.0, random_state=42, n_jobs=-1
)
voting = VotingRegressor(estimators=[('xgb', xgb_m), ('rf', rf)], weights=[2, 1])

models = {
    "Linear Regression (LogTarget)": TransformedTargetRegressor(regressor=Pipeline([("preprocessor", preprocessor), ("model", LinearRegression())]), func=np.log1p, inverse_func=np.expm1),
    "Random Forest (LogTarget)": TransformedTargetRegressor(regressor=Pipeline([("preprocessor", preprocessor), ("model", rf)]), func=np.log1p, inverse_func=np.expm1),
    "XGBoost (LogTarget)": TransformedTargetRegressor(regressor=Pipeline([("preprocessor", preprocessor), ("model", xgb_m)]), func=np.log1p, inverse_func=np.expm1),
    "Voting Ensemble (LogTarget)": TransformedTargetRegressor(regressor=Pipeline([("preprocessor", preprocessor), ("model", voting)]), func=np.log1p, inverse_func=np.expm1),
}

results = []
trained_pipelines = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    results.append(evaluate_model(name, y_test, preds))
    trained_pipelines[name] = model

results_df = pd.DataFrame(results)
print("--- Final Evaluation Results ---")
print(results_df.to_string())

--- Final Evaluation Results ---
                           Model          MAE          RMSE        R2
0  Linear Regression (LogTarget)  5371.849011  10559.618926  0.771830
1      Random Forest (LogTarget)  4723.849557   8567.337981  0.849806
2            XGBoost (LogTarget)  4678.976074   8582.003030  0.849291
3    Voting Ensemble (LogTarget)  4657.558276   8479.285329  0.852877


In [4]:
# Select Best Model and Serialize Pipeline
best_idx = results_df["R2"].idxmax()
best_model_name = results_df.loc[best_idx, "Model"]
best_pipeline = trained_pipelines[best_model_name]
print(f"Best Selected Model: {best_model_name}")

os.makedirs("../ml-service/model", exist_ok=True)
joblib.dump(best_pipeline, "../ml-service/model/best_model.pkl")

metadata = {
    "model_name": best_model_name,
    "target": "Rent",
    "metrics": results_df.to_dict(orient="records"),
    "features": list(X.columns)
}
joblib.dump(metadata, "../ml-service/model/model_metadata.pkl")
print("Model and metadata saved to ../ml-service/model/")

Best Selected Model: Voting Ensemble (LogTarget)
Model and metadata saved to ../ml-service/model/
